In [1]:
import pandas as pd
import numpy as np
# Load dataset
df = pd.read_csv(r"C:\\college_recommendation_system\\CleanData\\Cleaned_tgeapcet_2017_2024.csv")

df["category_gender"] = (
    df["category"].astype(str) + "_" + df["gender"].astype(str)
)

print(df.columns)
df.isnull().sum()
# Drop college_type and year_of_estab columns
df = df.drop(columns=["college_type", "year_of_estab"])
df.isnull().sum()
# Remove extra spaces
df["co_education"] = df["co_education"].str.strip()
df["category"] = df["category"].str.strip()

# Convert to uppercase
df["co_education"] = df["co_education"].str.upper()
df["category"] = df["category"].str.upper()

Index(['inst_code', 'institute_name', 'branch_code', 'branch_name',
       'co_education', 'college_type', 'year_of_estab', 'place', 'dist_code',
       'year', 'cutoff', 'category', 'gender', 'category_gender'],
      dtype='object')


In [2]:
from sklearn.preprocessing import LabelEncoder

le_college = LabelEncoder()
df["inst_code"] = le_college.fit_transform(df["inst_code"])

In [3]:
le_branch = LabelEncoder()
df["branch_code"] = le_branch.fit_transform(df["branch_code"])

In [4]:
df["category"] = df["category"].fillna("EWS")

category_map = {
    "OC":0,
    "BC_A":1,
    "BC_B":2,
    "BC_C":3,
    "BC_D":4,
    "BC_E":5,
    "SC":6,
    "ST":7,
    "EWS":8
}

df["category"] = df["category"].map(category_map)

In [5]:
gender_map = {
    "COED":0,
    "GIRLS":1
}
df["co_education"] = df["co_education"].map(gender_map)

In [6]:
le_location = LabelEncoder()
df["dist_code"] = le_location.fit_transform(df["dist_code"])

In [7]:
gender_map = {
    "Boys": 0,
    "Girls": 1
}

df["gender"] = df["gender"].map(gender_map)

In [8]:
df = df.drop(columns=[
    "institute_name",
    "branch_name",
    "category",
    "gender",
    "place"
])

In [9]:
df["category_gender"] = df["category_gender"].str.strip().str.upper()

In [10]:
from sklearn.preprocessing import LabelEncoder

le_catgen = LabelEncoder()
df["catgen_id"] = le_catgen.fit_transform(df["category_gender"])

In [11]:
df = df.drop(columns=["category_gender"])

In [12]:
df.head()

,inst_code,branch_code,co_education,dist_code,year,cutoff,catgen_id
0,0,20,0,2,2017,26907.0,12
1,0,30,0,2,2017,33871.0,12
2,0,33,0,2,2017,39938.0,12
3,0,46,0,2,2017,38774.0,12
4,1,13,0,11,2017,33769.0,12


In [16]:
# ==============================
# 1️⃣ Import Libraries
# ==============================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor


# ==============================
# 3️⃣ Sort Properly
# ==============================
df = df.sort_values(
    ["inst_code", "branch_code", "catgen_id", "year"]
).reset_index(drop=True)

# ==============================
# 4️⃣ Feature Engineering
# ==============================

# Previous year cutoff
df["prev_cutoff"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .shift(1)
)

# 2-year lag
df["prev_cutoff_2"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .shift(2)
)

# 3-year rolling mean
df["rolling_mean_3"] = (
    df.groupby(["inst_code", "branch_code", "catgen_id"])["cutoff"]
    .transform(lambda x: x.rolling(3).mean())
)

# Cutoff change
df["cutoff_change"] = df["cutoff"] - df["prev_cutoff"]

# Drop NaNs
df = df.dropna().reset_index(drop=True)

# ==============================
# 5️⃣ Define Features
# ==============================
features = [
    "inst_code",
    "branch_code",
    "co_education",
    "dist_code",
    # try removing year later if needed
    "year",
    "catgen_id",
    "prev_cutoff",
    "prev_cutoff_2",
    "rolling_mean_3",
    
]

# ==============================
# 6️⃣ Time-Based Split
# ==============================
train = df[df["year"] < 2024]
test = df[df["year"] == 2024]

X_train = train[features]
y_train = train["cutoff"]

X_test = test[features]
y_test = test["cutoff"]

# ==============================
# 7️⃣ Train XGBoost Model
# ==============================
model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ==============================
# 8️⃣ Predict
# ==============================
y_pred = model.predict(X_test)

# ==============================
# 9️⃣ Evaluation
# ==============================
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("XGBoost MAE:", mae)
print("XGBoost R2 (Accuracy):", r2)
print("Model Error: ",1-r2)

# ==============================
# 🔟 Save Predictions
# ==============================
results = test.copy()
results["predicted_cutoff"] = y_pred
results["error"] = results["cutoff"] - results["predicted_cutoff"]

results.to_csv("C:\\college_recommendation_system\\CleanData\\xgboost_cutoff_predictions_2024.csv", index=False)

print("Predictions saved to xgboost_cutoff_predictions_2024.csv")


XGBoost MAE: 7474.103729958992
XGBoost R2 (Accuracy): 0.9517620452587877
Model Error:  0.048237954741212286
Predictions saved to xgboost_cutoff_predictions_2024.csv


In [14]:
import pickle
model_path = "C:\\college_recommendation_system\\CleanData\\XG_Boost_cutoff_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully:", model_path)


# ==============================
# 🔟 Load Pickle Model
# ==============================

with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)


Model saved successfully: C:\college_recommendation_system\CleanData\XG_Boost_cutoff_model.pkl
